<a href="https://colab.research.google.com/github/Kafleavinash/Machinelearning_soil/blob/main/AI_foundry_2025_GEEMAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nbformat

# Load the notebook
nb = nbformat.read("/content/AI_foundry_2025_GEEMAP.ipynb", as_version=4)

# Clean invalid metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

# Save cleaned notebook
nbformat.write(nb, "cleaned_notebook.ipynb")


# **AI foundry 2025 - Geospatial AI in agriculture**


**Introduction to Google Earth Engine in Python**

- Earth Engine: <https://earthengine.google.com>
- Geemap: <https://geemap.org>
- samgeo: <https://samgeo.gishub.org/>

**Know the terminologies:**
- `Image`
- `ImageCollection()`
- `Feature`
- `FeatureCollection()`
- `geometry`



1. **Image**:
   - In Google Earth Engine, an image represents a single raster dataset, typically acquired from satellite or aerial sensors.
   - It consists of one or more bands, each containing pixel values representing various properties such as reflectance, temperature, or elevation.
   - Images can be single-date or multi-temporal, capturing data at different points in time.
   - Example: A Landsat satellite image showing the reflectance of the Earth's surface across different spectral bands.

2. **ImageCollection**:
   - An ImageCollection is a collection of images sharing similar properties or characteristics.
   - It allows users to organize and manipulate multiple images as a single entity, facilitating analysis and visualization tasks.
   - ImageCollections can represent datasets spanning large spatial and temporal extents, such as satellite imagery archives.
   - Example: A collection of Landsat images covering a specific region over multiple years.

3. **Feature**:
   - In Google Earth Engine, a feature represents a vector-based geospatial entity, such as a point, line, or polygon.
   - Features can have properties associated with them, storing additional information such as attribute values.
   - Features are used to represent spatial objects or phenomena, enabling spatial analysis and visualization.
   - Example: A point feature representing the location of a weather station, with properties like temperature and humidity.

4. **FeatureCollection**:
   - A FeatureCollection is a collection of features sharing similar geometry or attributes.
   - It allows users to manage and analyze multiple spatial features as a single dataset.
   - FeatureCollections are commonly used for storing and processing vector data, such as points, lines, and polygons.
   - Example: A collection of polygon features representing administrative boundaries of different countries.

5. **Geometry**:
   - Geometry refers to the spatial representation of features or objects in Earth Engine.
   - Geometries can be points, lines, polygons, or collections of these primitives.
   - Geometries define the shape, size, and location of spatial entities, enabling spatial analysis and visualization.
   - Example: The boundary of a national park represented as a polygon geometry, or the path of a river represented as a line geometry.

These concepts are fundamental in geospatial analysis and play a crucial role in Earth Engine workflows for data manipulation, visualization, and analysis.

================================================================================




**GOOGLE EARTH ENGINE - `geemap` hands-on learning**

**Import libraries**


In [ ]:
import ee   # Earth engine package
import geemap  # GEE for python

**Authentication**

Authenticate your google account. Once you run the below chunk, it will prompt you to put the authentication code from your google account.

This line is necessary because it initiates the **authentication process** with Google Earth Engine in order to access its services, including accessing and analyzing satellite data. This authentication step ensures that only authorized users can access Earth Engine resources and helps maintain the security of the platform.

In [ ]:
ee.Authenticate()

**Initialization of your project**

This line initializes the Earth Engine Python API with a specific project. By providing the project parameter, you are specifying which Google Cloud project to use for billing purposes and resource allocation. This step is essential for accessing Earth Engine resources within a specific project context, ensuring that usage is properly accounted for and billed to the appropriate project. The acedemic and research usage is **free**. It also allows you to access datasets and perform computations within the context of the specified project.

In [ ]:
ee.Initialize(project='ee-sunoj-aifoundry')

Initialize a map and display it

In [ ]:
m = geemap.Map(center = [39.89849869382563, -103.37899848234053], zoom = 12)
m.add_basemap('HYBRID')
m

Map(center=[39.89849869382563, -103.37899848234053], controls=(WidgetControl(options=['position', 'transparent…

**Adding surface elevation data**

In [ ]:
# Initializing the Google Earth Engine Map with specified center coordinates and zoom level
m = geemap.Map(center=[39.89849869382563, -103.37899848234053], zoom=12)

# Loading the Shuttle Radar Topography Mission (SRTM) Digital Elevation Model (DEM) image
image = ee.ImageCollection("USGS/3DEP/1m").mean()

# Visualization parameters for the SRTM image
vis_params = {
    "min": 0,                           # Minimum elevation value to map to the color palette
    "max": 6000,                        # Maximum elevation value to map to the color palette
    "palette": ["006633",               # Green
                "E5FFCC",               # Light green
                "662A00",               # Brown
                "D8D8D8",               # Light grey
                "F5F5F5"],              # White
}

# Adding the SRTM image layer to the map with the specified visualization parameters
m.add_layer(image, vis_params, "SRTM")

# Displaying the map
m


Map(center=[39.89849869382563, -103.37899848234053], controls=(WidgetControl(options=['position', 'transparent…

**Adding a satellite image (Sentinel-2A)**

https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED

In [ ]:
# Initializing the Google Earth Engine Map
m = geemap.Map()

# Filtering Sentinel-2 Surface Reflectance Harmonized data
# to select images between January 1, 2021, and January 1, 2022,
# with less than 5% cloud cover
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2021-01-01", "2022-01-01")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
)

# Generating a median composite image from the filtered collection
image = collection.median()

# Visualization parameters for the Sentinel-2 image
vis = {
    "min": 0.0,             # Minimum pixel value to map to the color palette
    "max": 3000,            # Maximum pixel value to map to the color palette
    "bands": ["B4", "B3", "B2"],  # Bands to use for visualization (R: B4, G: B3, B: B2)
    # "bands": ["B8", "B4", "B3"],  # Bands to use for visualization (NIR:B8, R: B4, G: B3)
}

# Setting the center of the map view to the specified coordinates and zoom level
m.set_center(-103, 40, 15)

# Adding the Sentinel-2 image layer to the map with the specified visualization parameters
m.add_layer(image, vis, "Sentinel-2")

# Displaying the map
m


Map(center=[40, -103], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

**Satellite imagery only on Champaign**

As I mentioned about the rich datasets that Google Earth Engine owns. We can use the boundary of Champaign County from the US Census Bureau's TIGER dataset. Since 'Champaign' county exists in other states, we should use a unique ID to filter out from the entire TIGER dataset. The unique id here is `GEOID == 17019`. Then we use `filterBounds()` to search for satellite imagery in the `ImageCollection()` only for Champaign county.

To display the image only for the Champaign county, we use `clipToCollection()` function to clip the image. This removes all the excess region.  

In [ ]:
# Initializing the Google Earth Engine Map
m = geemap.Map()

# Loading the US Counties dataset from the US Census Bureau (TIGER dataset, 2018 version)
fc = ee.FeatureCollection('TIGER/2018/Counties')

# Filtering the FeatureCollection to get the geometry of Champaign County, Illinois
champaignIL = fc.filter(ee.Filter.eq('GEOID', '53075'))

# Extracting the geometry of Champaign County, Illinois
champaignroi = champaignIL.geometry()

# Setting inputs for filtering Sentinel-2 data
start_month = '2019-06'  # Start date for data filtering
end_month = '2019-07'    # End date for data filtering

# Filtering Sentinel-2 Surface Reflectance Harmonized data
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(champaignIL)                      # Filter by the bounds of Champaign County, Illinois
    .filterDate(f'{start_month}-01', f'{end_month}-01')  # Filter by date range
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Filter images with less than 20% cloud cover
)

# Generating a median composite image from the filtered collection
image = collection.median()

# Clipping the image to the boundary of Champaign County, Illinois
image = image.clipToCollection(champaignIL)

# Visualization parameters for the cloud-free image
vis = {
    'min': 0.0,         # Minimum pixel value to map to the color palette
    'max': 3000,        # Maximum pixel value to map to the color palette
    'bands': ['B4', 'B3', 'B2'],  # Bands to use for visualization (R: B4, G: B3, B: B2)
}

# Adding the cloud-free image layer to the map
m.add_layer(image, vis, 'RGB image')

# Styling for the Champaign County boundary layer
style = {'color': 'FF0000FF', 'width': 4, 'lineType': 'solid', 'fillColor': 'FF000000'}

# Adding the Champaign County boundary layer to the map
m.add_layer(champaignIL.style(**style), {}, "Champaign county")

# Centering the map view on the geometry of Champaign County, Illinois
m.centerObject(champaignroi, 13)

# Displaying the map
m


Map(center=[46.90082859945265, -117.52311866405303], controls=(WidgetControl(options=['position', 'transparent…

**Normalized Difference Vegetation Index (NDVI)**

To the same chunk we will see how to add Normalized Difference Vegetation Index (NDVI) for the entire Champaign county. Do you recall the usage of NDVI? It is a plant health/growth indicator. If we visualize that layer, we will understand the planting stages across the county in a single chunk.

New portions added are:
- Calculating NDVI: `ndvi_img = image.normalizedDifference(['B8', 'B4'])`
- Adding visualization parameters: `ndvi_vis = {'min': 0.0, 'max': 1.0,'palette': 'turbo'}`
- Adding to the map: `m.add_layer(ndvi_img, ndvi_vis, 'NDVI image')`
- Adding a colorbar: `m.add_colorbar(ndvi_vis, label="NDVI", layer_name="NDVI image", orientation="vertical")`




In [ ]:
# Importing necessary libraries
import geemap
import ee

# Initializing the Google Earth Engine Map
m = geemap.Map()

# Loading the US Counties dataset from the US Census Bureau (TIGER dataset, 2018 version)
fc = ee.FeatureCollection('TIGER/2018/Counties')

# Filtering the FeatureCollection to get the geometry of Champaign County, Illinois
champaignIL = fc.filter(ee.Filter.eq('GEOID', '53075')) # This is for Whitman county, WA.

# Extracting the geometry of Champaign County, Illinois
champaignroi = champaignIL.geometry()

# Setting inputs for filtering Sentinel-2 data
start_month = '2019-06'  # Start date for data filtering
end_month = '2019-07'    # End date for data filtering

# Filtering Sentinel-2 Surface Reflectance Harmonized data
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(champaignIL)                      # Filter by the bounds of Champaign County, Illinois
    .filterDate(f'{start_month}-01', f'{end_month}-01')  # Filter by date range
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Filter images with less than 20% cloud cover
)

# Generating a median composite image from the filtered collection
image = collection.median()

# Clipping the image to the boundary of Champaign County, Illinois
image = image.clipToCollection(champaignIL)

# Calculating NDVI (Normalized Difference Vegetation Index)
ndvi_img = image.normalizedDifference(['B8', 'B4']) # (B8: NIR, B4: Red)

# Visualization parameters for the cloud-free image
vis = {
    'min': 0.0,         # Minimum pixel value to map to the color palette
    'max': 1500,        # Maximum pixel value to map to the color palette
    'bands': ['B4', 'B3', 'B2'],  # Bands to use for visualization (R: B4, G: B3, B: B2)
}

# Visualization parameters for the NDVI image
ndvi_vis = {
    'min': 0.0,         # Minimum pixel value to map to the color palette
    'max': 1.0,         # Maximum pixel value to map to the color palette
    'palette': 'turbo', # Color palette for visualization
}

# Adding the cloud-free image layer to the map
m.add_layer(image, vis, 'Cloud-free image')

# Adding the NDVI image layer to the map
m.add_layer(ndvi_img, ndvi_vis, 'NDVI image')

# Styling for the Champaign County boundary layer
style = {'color': 'FF0000FF', 'width': 4, 'lineType': 'solid', 'fillColor': 'FF000000'}

# Adding the Champaign County boundary layer to the map
m.add_layer(champaignIL.style(**style), {}, "Champaign")

# Adding colorbar to the NDVI image layer
m.add_colorbar(ndvi_vis, label="NDVI", layer_name="NDVI image", orientation="vertical")

# Centering the map view on the geometry of Champaign County, Illinois
m.centerObject(champaignroi, 13)

# Displaying the map
m


Map(center=[46.90082859945288, -117.52311866405309], controls=(WidgetControl(options=['position', 'transparent…


Before Google Earth Engine (GEE), downloading and visualizing this NDVI map would have required significant storage on the user's computer and hours to download gigabytes of data. With GEE, this step has been greatly simplified. One can quickly visualize the image. But it doesn't end there. You can perform a lot of analysis within GEE. Furthermore, with this Python integration, you can even run machine learning models directly in Colab or through Jupyter Notebook and utilize GPU computing on your own machine. If you are interested in learning more about setting up a virtual environment and running them on your local machine, please feel free to reach out to me, and I can assist you in setting up the environment.


**SENTINEL imagery with user-ROI**

In [ ]:
m = geemap.Map(center = [40,-88], zoom = 12)
m.add_basemap('HYBRID')
print("Draw an ROI")

m

Draw an ROI


Map(center=[40, -88], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

This chunk of code retrieves the region of interest (ROI) drawn by the user on the map (assuming m.user_roi contains user-defined ROI information), converts it into a feature collection, and then extracts its geometry. This geometry can then be used for further analysis or operations within Google Earth Engine.

In [ ]:
# Extracting the user-defined region of interest (ROI) from the map
userroi = ee.FeatureCollection(m.user_roi)

# Extracting the geometry of the user-defined ROI
roi = userroi.geometry()


The below chunk clips to user-ROi and produces NDVI image only for the drawn region.

In [ ]:
#======================================
# Inputs here:
# Setting the start and end months for filtering Sentinel-2 data
start_month = '2019-07'
end_month = '2019-08'
#======================================

# Filtering Sentinel-2 Surface Reflectance Harmonized data
# to select images within the user-defined region of interest (ROI),
# during the specified time period, and with less than 10% cloud cover
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(userroi)                              # Filter images within the user-defined ROI
    .filterDate(f'{start_month}-01', f'{end_month}-01') # Filter images by date range
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) # Filter images with less than 10% cloud cover
)

# Generating a median composite image from the filtered collection
image = collection.median()

# Clipping the image to the user-defined ROI
image = image.clipToCollection(userroi)

# Calculating NDVI (Normalized Difference Vegetation Index)
ndvi_img = image.normalizedDifference(['B8', 'B4'])

# Visualization parameters for the cloud-free image
vis = {
    'min': 0.0,                         # Minimum pixel value to map to the color palette
    'max': 1500,                        # Maximum pixel value to map to the color palette
    'bands': ['B4', 'B3', 'B2'],        # Bands to use for visualization (R: B4, G: B3, B: B2)
}

# Visualization parameters for the NDVI image
ndvi_vis = {
    'min': 0.0,                         # Minimum pixel value to map to the color palette
    'max': 1.0,                         # Maximum pixel value to map to the color palette
    'palette': 'turbo',                # Color palette for visualization
}

# Adding the cloud-free image layer to the map with specified visualization parameters
m.add_layer(image, vis, 'Cloud-free image')

# Adding the NDVI image layer to the map with specified visualization parameters
m.add_layer(ndvi_img, ndvi_vis, 'NDVI image')

# Styling for the user-defined ROI boundary layer
style = {'color': 'FF0000FF', 'width': 4, 'lineType': 'solid', 'fillColor': 'FF000000'}

# Adding the user-defined ROI boundary layer to the map
m.add_layer(userroi.style(**style), {}, "User_ROI")

# Adding colorbar to the NDVI image layer
m.add_colorbar(ndvi_vis, label="NDVI", layer_name="NDVI image", orientation="vertical")

# Centering the map view on the user-defined ROI
m.center_object(userroi)

# Displaying the map
m


Map(bottom=3175942.0, center=[40.00362113336921, -87.93326139450075], controls=(WidgetControl(options=['positi…

The provided color (t) is invalid. Using the default black color.
'#t' is not in web format. Need 3 or 6 hex digit.
The provided color (u) is invalid. Using the default black color.
'#u' is not in web format. Need 3 or 6 hex digit.
The provided color (r) is invalid. Using the default black color.
'#r' is not in web format. Need 3 or 6 hex digit.
The provided color (b) is invalid. Using the default black color.
'#b' is not in web format. Need 3 or 6 hex digit.
The provided color (o) is invalid. Using the default black color.
'#o' is not in web format. Need 3 or 6 hex digit.


In [ ]:
# geemap.ee_export_image(ndvi_img, filename="NDVI.tif", scale=10, region = roi)

Generating URL ...
Please wait ...
Data downloaded to /content/NDVI.tif


**National Agriculture Imagery Program (NAIP) imagery**

NAIP imagery, managed by the USDA, offers high-resolution aerial images covering the entire U.S. This data, typically refreshed every few years, aids in agricultural monitoring, land use planning, and precision farming. With resolutions as fine as 1 meter, it's widely used for crop identification, health assessments, and land cover analysis.


Now let's pull the high-resolution NAIP imagery into the GEE platform for the user-roi.

In [ ]:
m = geemap.Map(center = [40,-88], zoom = 12)
m.add_basemap('HYBRID')
print("Draw an ROI")
m

Draw an ROI


Map(center=[40, -88], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

In [ ]:
# Extracting the user-defined region of interest (ROI) from the map
userroi = ee.FeatureCollection(m.user_roi)

# Extracting the geometry of the user-defined ROI
roi = userroi.geometry()


In [ ]:
#======================================
# Inputs
year = '2017'
#======================================

# Loading NAIP dataset for the specified year and region of interest (ROI)
dataset = (
    ee.ImageCollection('USDA/NAIP/DOQQ')
    .filter(ee.Filter.date(f'{year}-01-01', f'{year}-12-31'))
    .filterBounds(roi)
)

NAIP_all_bands = dataset.mean().clip(roi)

# Generating False Color composite
FalseColor = dataset.select(['N', 'R', 'G']).mean().clip(roi)
# Calculating NDVI from False Color composite
ndvi_img = FalseColor.normalizedDifference(['N', 'R'])

# Generating True Color composite
TrueColor = dataset.select(['R', 'G', 'B']).mean().clip(roi)

# Visualization parameters
ColorVis = {'min': 0, 'max': 255}
ndvi_vis = {'min': 0, 'max': 1, 'palette': 'turbo'}

# Adding layers to the map
m.addLayer(FalseColor, ColorVis, 'False color')
m.addLayer(TrueColor, ColorVis, 'True color')
m.addLayer(ndvi_img, ndvi_vis, 'NDVI')

# Adding color bar for NDVI
m.add_colorbar(ndvi_vis, label="NDVI", layer_name="NDVI", orientation="horizontal")

# Centering the map view on the region of interest (ROI)
m.center_object(roi)

# Displaying the map
m


Map(bottom=6351710.204636298, center=[40.001546719316956, -87.8406034928112], controls=(WidgetControl(options=…

## Downloading image from small ROI

In [ ]:
geemap.ee_export_image(TrueColor, filename="RGB.tif", scale=1, region = roi)

Generating URL ...
Please wait ...
Data downloaded to /content/RGB.tif


## Downloading Satellite Imagery using tiles

Sometimes for your project, you might need to download imagery for a specific region. The chunk below initializes a basemap, allowing the user to draw a region of interest. We can then retrieve the imagery for that region. If you are satisfied with the satellite imagery, we will proceed to download it in the next few code chunks. However, if the region is large, it may exceed the maximum download capacity. In such cases, you can still download the imagery by splitting it into grids using the `fishnet` function, and then proceed with the download.

In [ ]:
# Centering the map view on the region of interest (ROI)
m.centerObject(roi)

# Creating a fishnet grid over the region of interest
grid = geemap.fishnet(roi, rows=2, cols=2)

# Adding the fishnet grid as a layer to the map
m.addLayer(grid, {}, "Grids")
m

Map(bottom=12703080.0, center=[40.00188293458077, -87.84420669078827], controls=(WidgetControl(options=['posit…

In [ ]:
# Setting the output directory path where data will be saved
out_dir = "/content/AIFoundry_2025"
out_dir


'/content/AIFoundry_2025'

In [ ]:
# Sometimes while downloading, there is an error showing "No module named 'geedim'"
# In that case uncomment the below line and run the chunk

!pip install geedim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 21.1 MB/s eta 0:00:00


In [ ]:
# Downloading Earth Engine image tiles for NDVI image
# Parameters:
# ndvi_img: The NDVI image to download tiles for
# grid: The grid (e.g., fishnet) defining the tiles
# out_dir: The output directory where tiles will be saved
# prefix: Prefix to add to each downloaded tile file
# crs: The coordinate reference system (CRS) of the tiles
# scale: The scale of the tiles

geemap.download_ee_image_tiles(
    TrueColor, grid, out_dir, prefix="NAIP_RGB_", crs="EPSG:4326", scale=10
)


NAIP_RGB_1.tif: |          | 0.00/98.3k (raw) [  0.0%] in 00:00 (eta:     ?)

There is no STAC entry for: None


NAIP_RGB_2.tif: |          | 0.00/98.3k (raw) [  0.0%] in 00:00 (eta:     ?)

NAIP_RGB_3.tif: |          | 0.00/98.3k (raw) [  0.0%] in 00:00 (eta:     ?)

NAIP_RGB_4.tif: |          | 0.00/96.1k (raw) [  0.0%] in 00:00 (eta:     ?)

...

NAIP_RGB_6.tif: |          | 0.00/96.1k (raw) [  0.0%] in 00:00 (eta:     ?)

NAIP_RGB_7.tif: |          | 0.00/98.3k (raw) [  0.0%] in 00:00 (eta:     ?)

NAIP_RGB_8.tif: |          | 0.00/98.3k (raw) [  0.0%] in 00:00 (eta:     ?)

NAIP_RGB_9.tif: |          | 0.00/98.3k (raw) [  0.0%] in 00:00 (eta:     ?)

Downloaded 9 tiles in 18.236598014831543 seconds.


=====================================================================

**ZONAL STATISTICS**

Zonal statistics is a technique used in geospatial image analysis to calculate summary statistics for raster data (image) within predefined zones or regions defined by vector polygons (boundaries). It helps to quantify the characteristics of raster data within specific geographic areas.

With raster and vector layers, zonal statistics involves the following steps:

1. **Raster Layer**: The raster layer contains the data for which you want to calculate statistics. This could be satellite imagery, elevation, land cover, temperature, etc.

2. **Vector Layer**: The vector layer defines the zones or regions of interest. These could be state boundaries, county, or any other defined areas.

3. **Zonal Statistics Calculation**: Zonal statistics are then calculated for each zone defined by the vector layer. Common statistics include mean, median, sum, standard deviation, minimum, maximum, etc.

4. **Output**: The output is typically a table where each row corresponds to a zone (states, grid), and the columns contain the calculated statistics for each zone.

For example, you might use zonal statistics to calculate the average temperature within each grid, based on temperature data from a raster layer.


In [ ]:
# Create a new map centered at latitude 40 and longitude -100, with a zoom level of 5
Map = geemap.Map(center=[40, -100], zoom=5)

# Retrieve the first image from the NOAA/GFS0P25 ImageCollection and select the temperature_2m_above_ground band
collection = (
    ee.ImageCollection("NOAA/GFS0P25")
    .filterDate("2025-06-04", "2025-06-05")
    .select("temperature_2m_above_ground")
)

image = collection.mean()

# Define visualization parameters for the temperature image
vis_params = {
    "min": -40.0,                           # Minimum temperature value to map to the color palette
    "max": 40.0,                            # Maximum temperature value to map to the color palette
    "palette": ["blue", "purple", "cyan",   # Color palette for visualization
                "green", "yellow", "red"]
}

# Add the temperature image layer to the map with specified visualization parameters
Map.addLayer(image, vis_params, "Temperature")

# Add a color bar to the map to visualize temperature values, with label "Temperature (°C)"
Map.add_colorbar(vis_params, label="Temperature (°C)")

# Display the map
Map


Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

## JavaScript to Python

In [ ]:
Map = geemap.Map(center = [40, -88], zoom = 16)
Map.add_basemap('HYBRID')
Map,go_to_setting_handler_then_converter

Map(center=[40, -88], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

<IPython.core.display.Javascript object>

In [ ]:
# The code has been copied to the clipboard.
# Press Ctrl+V to in a code cell to paste it.
dataset = ee.ImageCollection('NOAA/GFS0P25') \
.filter(ee.Filter.date('2025-05-01', '2025-05-02'))
temperatureAboveGround = dataset.select('temperature_2m_above_ground')
visParams = {
    "min": -40.0,
    "max": 35.0,
    "palette": ['blue', 'purple', 'cyan', 'green', 'yellow', 'red'],
}


Map.setCenter(-88, 40, 16)
Map.addLayer(temperatureAboveGround, visParams, 'Temperature Above Ground')
Map


Map(bottom=972.0, center=[40, -88], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sea…

In [ ]:
# Define a bounding box geometry with coordinates (-130, 24, -68, 50)
# geometry = ee.Geometry.BBox(-130, 24, -68, 50)

userroi = ee.FeatureCollection(Map.user_roi)
geometry = userroi.geometry()

# Create a grid with a specified grid size (200,000 meters) using the bounding box geometry
grid = geemap.create_grid(geometry, 100000)

# Add the grid as a layer to the map with default visualization parameters
Map.addLayer(grid, {}, "Grid")


In [ ]:
# Calculate zonal statistics for the image within each grid cell
# Parameters:
# image: The image for which zonal statistics will be calculated
# grid: The grid defining the zones or regions of interest
# stat_type: The type of statistic to calculate (e.g., "MEAN", "MAX", "MIN") MEAN, MAXIMUM, MINIMUM, MEDIAN, STD, MIN_MAX, VARIANCE, SUM
# scale: The scale (in meters) at which to perform the calculation
# return_fc: Whether to return the results as a FeatureCollection

stats = geemap.zonal_stats(image, grid, stat_type="MEAN", scale=1e5, return_fc=True)
stats

Computing statistics ...


In [ ]:
Map.add_styled_vector(
    stats, column="mean", palette="coolwarm", layer_name="Mean Temperature"
)
Map.add_layer_manager()
Map

Map(bottom=12634.0, center=[40.59727063442027, -88.472900390625], controls=(WidgetControl(options=['position',…

**ZONAL STATISTICS - Elevation and Satellite image**

In [ ]:
Map = geemap.Map(center = [40,-100], zoom= 5)
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [ ]:
# Add Earth Engine Digital Elevation Model (DEM) dataset
dem = ee.Image("USGS/SRTMGL1_003")

# Set visualization parameters for DEM
dem_vis = {
    "min": 0,                                    # Minimum elevation value to map to the color palette
    "max": 4000,                                 # Maximum elevation value to map to the color palette
    "palette": ["006633", "E5FFCC", "662A00",   # Color palette for visualization
                "D8D8D8", "F5F5F5"]
}

# Add DEM to the map
Map.addLayer(dem, dem_vis, "SRTM DEM")

#===========================================================
# Add Landsat data to the map
landsat = ee.Image("LANDSAT/LE7_TOA_5YEAR/1999_2003")

# Set visualization parameters for Landsat
landsat_vis = {"bands": ["B4", "B3", "B2"],     # Bands to visualize (RGB)
               "gamma": 1.4}                    # Gamma correction value

# Add Landsat image to the map
Map.addLayer(landsat, landsat_vis, "Landsat 5-year TOA 1999-2003")

#===========================================================
# Add US States boundary to the map
states = ee.FeatureCollection("TIGER/2018/States")

# Add US States boundary to the map
Map.addLayer(states, {}, "US States")

# Display the map
Map

Map(bottom=3401.0, center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=S…

**Zonal statistics with elevation layer**

In [ ]:
import os

# Setting the output directory path where data will be saved
out_dir = "/content/AIFoundry_2025"

# Displaying the output directory path
# out_dir

# Defining the output file path for storing zonal statistics as a CSV file
out_dem_stats = os.path.join(out_dir, "dem_stats.csv")

# Allowed output formats: csv, shp, json, kml, kmz
# Allowed statistics types: MEAN, MAXIMUM, MINIMUM, MEDIAN, STD, MIN_MAX, VARIANCE, SUM
# Calculating zonal statistics for the DEM image within each state boundary polygon
geemap.zonal_stats(dem, states, out_dem_stats, stat_type="MEAN", scale=1000)


Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /content/AIFoundry_2025/dem_stats.csv


**Zonal statistics for Landsat imagery**

In [ ]:
out_landsat_stats = os.path.join(out_dir, "landsat_stats.csv")
geemap.zonal_stats(landsat, states, out_landsat_stats, stat_type="MEAN", scale=1000)

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /content/AIFoundry_2025/landsat_stats.csv


**Dataset generation for ML model**

For yield prediction modeling, you might use zonal statistics on a field level. Lets draw an ROI for a field and collect sentinel NDVI data only for that field and save it as CSV.  


In [ ]:
Map = geemap.Map(center = [40,-88], zoom= 14)
Map.add_basemap('HYBRID')
print("Draw an ROI")
Map

Draw an ROI


Map(center=[40, -88], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

In [ ]:
# Extracting the user-defined region of interest (ROI) from the map
userroi = ee.FeatureCollection(Map.user_roi)

# Extracting the geometry of the user-defined ROI
roi = userroi.geometry()

# Create a grid with a specified grid size (30 meters) using the bounding box geometry
grid = geemap.create_grid(roi, 30)

# Add the grid as a layer to the map with default visualization parameters
Map.addLayer(grid, {}, "Grid")
Map


Map(bottom=3175562.0, center=[40.016111851185435, -87.99224853515626], controls=(WidgetControl(options=['posit…

In [ ]:
#======================================
# Inputs here:
# Setting the start and end months for filtering Sentinel-2 data
start_month = '2023-07'
end_month = '2023-08'
#======================================

# Filtering Sentinel-2 Surface Reflectance Harmonized data
# to select images within the user-defined region of interest (ROI),
# during the specified time period, and with less than 10% cloud cover
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(userroi)                              # Filter images within the user-defined ROI
    .filterDate(f'{start_month}-01', f'{end_month}-01') # Filter images by date range
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) # Filter images with less than 10% cloud cover
)

# Generating a median composite image from the filtered collection
image = collection.median()

# Clipping the image to the user-defined ROI
image = image.clipToCollection(userroi)

# Calculating NDVI (Normalized Difference Vegetation Index)
ndvi_img = image.normalizedDifference(['B8', 'B4'])

# Visualization parameters for the cloud-free image
vis = {
    'min': 0.0,                         # Minimum pixel value to map to the color palette
    'max': 1500,                        # Maximum pixel value to map to the color palette
    'bands': ['B4', 'B3', 'B2'],        # Bands to use for visualization (R: B4, G: B3, B: B2)
}

# Visualization parameters for the NDVI image
ndvi_vis = {
    'min': 0.0,                         # Minimum pixel value to map to the color palette
    'max': 1.0,                         # Maximum pixel value to map to the color palette
    'palette': 'turbo',                # Color palette for visualization
}

# Adding the cloud-free image layer to the map with specified visualization parameters
Map.add_layer(image, vis, 'Cloud-free image')

# Adding the NDVI image layer to the map with specified visualization parameters
Map.add_layer(ndvi_img, ndvi_vis, 'NDVI image')

# Styling for the user-defined ROI boundary layer
style = {'color': 'FF0000FF', 'width': 4, 'lineType': 'solid', 'fillColor': 'FF000000'}

# Adding the user-defined ROI boundary layer to the map
Map.add_layer(userroi.style(**style), {}, "User_ROI")

# Adding colorbar to the NDVI image layer
Map.add_colorbar(ndvi_vis, label="NDVI", layer_name="NDVI image", orientation="vertical")

# Centering the map view on the user-defined ROI
Map.center_object(userroi)

# Add the grid as a layer to the map with default visualization parameters
Map.addLayer(grid, {}, "Grid")

# Displaying the map
Map


Map(bottom=3175562.0, center=[40.016111851185435, -87.99224853515626], controls=(WidgetControl(options=['posit…

In [ ]:
import os

# Setting the output directory path where data will be saved
out_dir = "/content/AIFoundry_2025"

# Displaying the output directory path
out_dir

# Defining the output file path for storing zonal statistics as a CSV file
ndvi_stats = os.path.join(out_dir, "ndvi_stats.csv")

# Calculating zonal statistics for the NDVI image within each grid cell
geemap.zonal_stats(ndvi_img, grid, ndvi_stats, stat_type="MEAN", scale=10)

Computing statistics ...
Generating URL ...
Please wait ...
Data downloaded to /content/AIFoundry_2025/ndvi_stats.csv


## Load your own shapefile from GoogleDrive

In [ ]:
m = geemap.Map(center = [40,-88], zoom= 14)
m.add_basemap('HYBRID')
print("Draw an ROI")
m


Draw an ROI


Map(center=[40, -88], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(chi…

In [ ]:
!pip install PyCRS

  Preparing metadata (setup.py) ... done
  Created wheel for PyCRS: filename=PyCRS-1.0.2-py3-none-any.whl size=32686 sha256=0ae980c5a84b427454b0531ed1d3409bea2c6f502208d3d647fd4dad894c85d0
  Stored in directory: /root/.cache/pip/wheels/5f/ad/a3/183ed754d7698fc15a2eb153705e05d05a0d97f3331293ce48
Successfully built PyCRS


In [ ]:
path = "path_to_your_shapefile"

fc = geemap.shp_to_ee(path)
m.addLayer(fc, {}, "Multiple_fields")
m.center_object(fc, 14)
m



Map(bottom=1588176.0, center=[40.14862057742017, -88.07634988113642], controls=(WidgetControl(options=['positi…

You can use this CSV and zonal statistics from other data layers to build ML models with geospatial data.

================================================================================